# Practice Test 1

**Set a 120-minute timer before you start.**

Rules:
- Implement everything from scratch (no copying from practice notebooks)
- NumPy/Python docs are allowed
- No AI tools, no pasting external code

**Structure:**
- Q1: Softmax family (single)
- Q2: einsum batch operations (single)
- Q3: Attention — 3 parts, unlock sequentially
- Q4: Transformer components — 3 parts, last one harder


In [ ]:
import numpy as np

def softmax(z, axis=-1):
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def check(name, got, expected=None, cond=None, rtol=1e-4, atol=1e-6):
    try:
        if cond is not None:
            passed = bool(cond(got))
        elif expected is not None:
            passed = np.allclose(np.array(got, dtype=float),
                                 np.array(expected, dtype=float), rtol=rtol, atol=atol)
        else:
            passed = True
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed and expected is not None:
            g, e = np.array(got), np.array(expected)
            print(f"         got shape {g.shape}, expected shape {e.shape}")
            if g.shape == e.shape and g.size <= 16:
                print(f"         got:      {g.ravel()}")
                print(f"         expected: {e.ravel()}")
        return passed
    except Exception as exc:
        print(f"  [ERROR] {name}: {type(exc).__name__}: {exc}")
        return False

print("Helper loaded. softmax and check() are available.")


## Q1: Softmax Family

In [ ]:
def softmax(z, axis=-1):
    raise NotImplementedError()

def log_softmax(z, axis=-1):
    """Do NOT compute softmax first then log."""
    raise NotImplementedError()

def cross_entropy(logits, targets):
    """logits: (B,C). targets: (B,) ints. Returns scalar mean loss."""
    raise NotImplementedError()


In [ ]:
print("="*50 + "\nQ1 TESTS\n" + "="*50)
# softmax
check("softmax sums to 1", softmax(np.array([1.,2.,3.])).sum(), 1.0)
check("softmax stable (z=[1000,1001,1002])",
      cond=lambda x: np.all(np.isfinite(x)), got=softmax(np.array([1000.,1001.,1002.])))
check("softmax 2D rows sum to 1",
      softmax(np.random.randn(4,6), axis=-1).sum(axis=-1), np.ones(4))
# log_softmax
z = np.array([1.,2.,3.])
check("log_softmax = log(softmax)", log_softmax(z), np.log(softmax(z)), rtol=1e-5)
check("log_softmax stable on large z",
      cond=lambda x: np.all(np.isfinite(x)), got=log_softmax(np.array([1000.,1001.,1002.])))
# cross_entropy
logits = np.array([[5.,0.,0.],[0.,5.,0.],[0.,0.,5.]])
targets = np.array([0,1,2])
check("low loss on correct preds", cond=lambda l: l < 0.1, got=cross_entropy(logits, targets))
check("uniform → log(C)", cross_entropy(np.zeros((4,3)), np.array([0,1,2,0])), np.log(3), rtol=1e-4)
check("scalar output", cond=lambda l: np.array(l).ndim == 0, got=cross_entropy(logits, targets))


## Q2: einsum Batch Operations

In [ ]:
def batch_outer_products(A, B):
    """A: (B,m). B: (B,n). Returns (B,m,n). Single einsum call."""
    raise NotImplementedError()

def weighted_value_sum(weights, V):
    """weights: (B,H,T,T). V: (B,H,T,d_v). Returns (B,H,T,d_v). Single einsum call."""
    raise NotImplementedError()

def virtual_ov(W_OV_L1, W_OV_L2):
    """W_OV_L1/L2: (d_model, d_model). Returns W_OV_L2 @ W_OV_L1 via einsum."""
    raise NotImplementedError()


In [ ]:
print("="*50 + "\nQ2 TESTS\n" + "="*50)
B, m, n = 4, 3, 5
A = np.random.randn(B,m); Bm = np.random.randn(B,n)
res = batch_outer_products(A, Bm)
check("outer shape (B,m,n)", cond=lambda x: x.shape == (B,m,n), got=res)
check("outer [0] matches np.outer", res[0], np.outer(A[0], Bm[0]))

bsz, H, T, dv = 2, 4, 6, 8
weights = softmax(np.random.randn(bsz,H,T,T), axis=-1)
V = np.random.randn(bsz,H,T,dv)
out = weighted_value_sum(weights, V)
check("weighted sum shape (B,H,T,dv)", cond=lambda x: x.shape == (bsz,H,T,dv), got=out)
check("weighted sum [0,0,0] matches manual", out[0,0,0], weights[0,0,0] @ V[0,0])

d = 8
OV1 = np.random.randn(d,d); OV2 = np.random.randn(d,d)
check("virtual_ov = OV2 @ OV1", virtual_ov(OV1, OV2), OV2 @ OV1)


## Q3: Attention (3 Parts — unlock sequentially)

In [ ]:
# Part 1: Scaled dot-product attention, no mask
def sdp_attention(Q, K, V):
    """Q: (T,dk). K: (T,dk). V: (T,dv). Returns output (T,dv), weights (T,T)."""
    raise NotImplementedError()


In [ ]:
print("Q3 Part 1 Tests")
T, dk, dv = 6, 4, 8
Q = np.random.randn(T,dk); K = np.random.randn(T,dk); V = np.random.randn(T,dv)
out, w = sdp_attention(Q, K, V)
check("output shape (T,dv)", cond=lambda x: x.shape == (T,dv), got=out)
check("weights shape (T,T)", cond=lambda x: x.shape == (T,T), got=w)
check("weights sum to 1", w.sum(axis=-1), np.ones(T), atol=1e-5)
check("scaled by sqrt(dk)", cond=lambda x: 0.2 < np.std(x) < 3.0, got=w)
print("  --> Pass Part 1 before continuing")


In [ ]:
# Part 2: Add causal mask
def causal_sdp_attention(X, W_Q, W_K, W_V):
    """X: (T,dm). Returns output (T,dv), weights (T,T) with causal mask."""
    raise NotImplementedError()


In [ ]:
print("Q3 Part 2 Tests")
T, dm, dk, dv = 7, 12, 4, 6
X = np.random.randn(T,dm)
W_Q = np.random.randn(dm,dk); W_K = np.random.randn(dm,dk); W_V = np.random.randn(dm,dv)
out, w = causal_sdp_attention(X, W_Q, W_K, W_V)
check("output shape (T,dv)", cond=lambda x: x.shape == (T,dv), got=out)
check("upper triangle ~0", cond=lambda x: np.allclose(np.triu(x,k=1), 0, atol=1e-6), got=w)
check("pos 0 attends to self only", cond=lambda x: np.isclose(x[0,0], 1.0), got=w)
print("  --> Pass Part 2 before continuing")


In [ ]:
# Part 3: Multi-head
def mha(X, W_Qs, W_Ks, W_Vs, W_O, causal=True):
    """X: (T,dm). W_Qs/Ks/Vs: (H,dm,dk or dv). W_O: (H*dv,dm). Returns (T,dm)."""
    raise NotImplementedError()


In [ ]:
print("Q3 Part 3 Tests")
T, dm, H, dk, dv = 5, 16, 4, 4, 4
X = np.random.randn(T,dm)
W_Qs = np.random.randn(H,dm,dk); W_Ks = np.random.randn(H,dm,dk)
W_Vs = np.random.randn(H,dm,dv); W_O = np.random.randn(H*dv,dm)
out = mha(X, W_Qs, W_Ks, W_Vs, W_O)
check("output shape (T,dm)", cond=lambda x: x.shape == (T,dm), got=out)
check("non-trivial output", cond=lambda x: np.std(x) > 0.01, got=out)


## Q4: Transformer Components (3 Parts — last one harder)

In [ ]:
# Part 1: Building blocks
def layer_norm(x, eps=1e-5):
    raise NotImplementedError()

def gelu(x):
    raise NotImplementedError()

def mlp_block(x, W1, W2):
    """x: (T,dm). W1: (dm,dff). W2: (dff,dm). Returns (T,dm)."""
    raise NotImplementedError()


In [ ]:
print("Q4 Part 1 Tests")
x = np.random.randn(4, 8)
ln = layer_norm(x)
check("layernorm mean ~0", ln.mean(axis=-1), np.zeros(4), atol=1e-6)
check("layernorm var ~1", ln.var(axis=-1), np.ones(4), atol=1e-4)
check("gelu(0)=0", gelu(np.array([0.])), np.array([0.]), atol=1e-6)
W1 = np.random.randn(8,32); W2 = np.random.randn(32,8)
check("mlp shape (T,dm)", cond=lambda o: o.shape == (4,8), got=mlp_block(x, W1, W2))


In [ ]:
# Part 2: Full transformer layer (pre-norm, multi-head)
def transformer_layer(x, W_Qs, W_Ks, W_Vs, W_O, W1, W2, causal=True):
    """One full pre-norm transformer layer. x: (T,dm). Returns (T,dm)."""
    raise NotImplementedError()


In [ ]:
print("Q4 Part 2 Tests")
T, dm, H, dk, dv, dff = 5, 16, 2, 4, 4, 32
x = np.random.randn(T, dm)
W_Qs = np.random.randn(H,dm,dk); W_Ks = np.random.randn(H,dm,dk)
W_Vs = np.random.randn(H,dm,dv); W_O = np.random.randn(H*dv,dm)
W1 = np.random.randn(dm,dff);   W2 = np.random.randn(dff,dm)
out = transformer_layer(x, W_Qs, W_Ks, W_Vs, W_O, W1, W2)
check("output shape (T,dm)", cond=lambda o: o.shape == (T,dm), got=out)
check("output ≠ input", cond=lambda o: not np.allclose(o, x), got=out)
check("output finite", cond=lambda o: np.all(np.isfinite(o)), got=out)


In [ ]:
# Part 3 (Harder): Attention entropy analysis
def mean_attention_entropy(weights_list):
    """
    weights_list: list of (T,T) attention weight matrices (one per head).
    Entropy of row t: H(t) = -sum(w * log(w + 1e-10))
    Returns (H,) array of mean entropy across token positions.
    """
    raise NotImplementedError()

def most_uniform_head(weights_by_layer):
    """
    weights_by_layer: list of n_layers lists, each containing H (T,T) matrices.
    Returns (layer_idx, head_idx) of the head with highest mean entropy.
    """
    raise NotImplementedError()


In [ ]:
print("Q4 Part 3 Tests")
T, n_layers, H = 8, 3, 4
np.random.seed(42)
attn_layers = []
for l in range(n_layers):
    heads = []
    for h in range(H):
        if l == 1 and h == 2:  # planted: uniform
            w = np.ones((T,T)) / T
        else:
            w = softmax(np.random.randn(T,T)*3, axis=-1)
        heads.append(w)
    attn_layers.append(heads)
ents = mean_attention_entropy(attn_layers[0])
check("entropy shape (H,)", cond=lambda x: x.shape == (H,), got=ents)
check("uniform head entropy = log(T)", mean_attention_entropy(attn_layers[1])[2], np.log(T), rtol=0.05)
li, hi = most_uniform_head(attn_layers)
check("finds layer 1", li, 1)
check("finds head 2", hi, 2)
